In [1]:
import os
import zipfile
import pandas as pd
from google.colab import files
zip_path = "/content/DEPI-GRADUATION-PROJECT-main (1).zip"
if os.path.exists(zip_path):
    print("ZIP file found!")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content")

    print("Extraction completed!")
else:
    print("ZIP file not found!")

ZIP file found!
Extraction completed!


In [2]:
base_path = "/content/DEPI-GRADUATION-PROJECT-main"
print("Project folder exists:", os.path.isdir(base_path))

if os.path.isdir(base_path):
    print("\nProject contents:")
    print(os.listdir(base_path))

Project folder exists: True

Project contents:
['README.md', 'data', 'src', 'task2.ipynb', '.gitignore', 'docs', 'notebooks', 'requirements.txt']


In [3]:

task7_path = os.path.join(base_path, "data", "processed", "task7")
print("Task 7 folder exists:", os.path.isdir(task7_path))

if os.path.isdir(task7_path):
    print("\nFiles:")
    for file in os.listdir(task7_path):
        print(file)

Task 7 folder exists: True

Files:
eval_df.csv
train_model_data.csv


In [4]:
eval_path = os.path.join(task7_path, "eval_df.csv")
train_path = os.path.join(task7_path, "train_model_data.csv")
eval_df = pd.read_csv(eval_path)
train_model_data = pd.read_csv(train_path)
print("Evaluation shape:", eval_df.shape)
print("Training shape:", train_model_data.shape)
print("\nEvaluation columns:")
print(eval_df.columns.tolist())
print("\nTraining columns:")
print(train_model_data.columns.tolist())

Evaluation shape: (494, 77)
Training shape: (5048, 71)

Evaluation columns:
['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'is_phone_listing', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known', 'storage_imp_src', 'storage_imp_conf', 'storage_imp_post', 'ram_imp_src', 'ram_imp_conf', 'ram_imp_post', 'imp_price_used', 'ref_matched', 'storage_gb_orig', 'storage_gb_was_imputed', 'ram_gb_orig', 'ram_gb_was_imputed', 'ref_n', 'ref_med_log', 'ref_mad_log', 'ref_level', 'ref_median_price',

In [5]:
# Evaluation rows with valid actual and predicted prices
report_df = eval_df[
    (eval_df['listing_type'] == 'phone') &
    eval_df['price'].notna() &
    eval_df['predicted_price'].notna() &
    eval_df['absolute_error'].notna()
].copy()

print("Total evaluated phone listings:", len(report_df))
# 1. MAE by Brand
brand_report = report_df.groupby('brand_group', dropna=False).agg(
    ads_count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean')
).reset_index().sort_values('MAE')

print("\n--- MAE by Brand ---")
display(brand_report)

# 2. MAE by Price Band
price_report = report_df.groupby('price_band', dropna=False).agg(
    ads_count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean')
).reset_index().sort_values('MAE')

print("\n--- MAE by Price Band ---")
display(price_report)

# 3. MAE by Model Frequency Band
frequency_report = report_df.groupby('model_frequency_band', dropna=False).agg(
    ads_count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean')
).reset_index().sort_values('MAE')

print("\n--- MAE by Model Frequency Band ---")
display(frequency_report)

Total evaluated phone listings: 494

--- MAE by Brand ---


,brand_group,ads_count,MAE
2,honor,25,2904.053508
10,vivo,9,3110.779108
5,motorola,5,4548.685244
11,xiaomi,66,4626.408680
4,infinix,14,4659.806504
6,oppo,19,4720.374816
3,huawei,7,5726.095177
8,samsung,210,6574.210856
7,realme,15,7074.609593
0,apple,122,7752.860529



--- MAE by Price Band ---


,price_band,ads_count,MAE
0,Budget (<5k),70,2928.020509
2,Mid-range (10k-20k),88,4238.287749
1,Lower-mid (5k-10k),111,5066.129728
4,Upper-mid (20k-40k),100,5975.553575
3,Premium (40k+),125,10801.838363



--- MAE by Model Frequency Band ---


,model_frequency_band,ads_count,MAE
4,Unseen (0),48,5097.661605
2,Rare (1-9),113,5464.421053
3,Unknown model key,12,6391.948808
1,Low (10-29),138,6406.562493
0,Frequent (30+),183,6913.005270


In [6]:
# Identify unseen models
unseen_df = report_df[
    (report_df['train_model_count'] == 0) &
    (report_df['model_key_norm'].notna()) &
    (report_df['model_key_norm'].astype(str).str.strip().str.lower() != 'unknown')
].copy()

print("Unseen-model listings:", len(unseen_df))
print("Distinct unseen models:", unseen_df['model_key_norm'].nunique())

print("\nOverall Unseen-model MAE:")
print(unseen_df['absolute_error'].mean())

# MAE by unseen model
unseen_model_report = unseen_df.groupby('model_key_norm').agg(
    ads_count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    median_actual_price=('price', 'median'),
    median_predicted_price=('predicted_price', 'median')
).reset_index().sort_values('MAE', ascending=False)

print("\n--- MAE by Unseen Model ---")
display(unseen_model_report)

# Worst 20 unseen listings
print("\n--- Worst 20 Unseen Listings ---")

worst_unseen = unseen_df.sort_values(
    'absolute_error', ascending=False
)[[
    'title',
    'brand',
    'model_key_norm',
    'price',
    'predicted_price',
    'absolute_error'
]].head(20)

display(worst_unseen)

Unseen-model listings: 48
Distinct unseen models: 38

Overall Unseen-model MAE:
5097.661604662572

--- MAE by Unseen Model ---


,model_key_norm,ads_count,MAE,median_actual_price,median_predicted_price
5,huawei mate x5,1,26047.485499,50000.0,23952.514501
25,sony xperia pro-i,1,20878.158082,45543.0,24664.841918
29,xiaomi 8 pro,1,13038.806585,6000.0,19038.806585
11,infinix note 60,2,11351.716292,19250.0,7898.283708
1,google pixel 9 pro,1,10933.916259,32000.0,21066.083741
19,samsung a27,1,10310.114984,18000.0,7689.885016
10,infinix note 5,1,10247.194674,12500.0,2252.805326
16,oppo r15 pro,1,7722.234325,1000.0,8722.234325
23,samsung grand,1,6960.923477,1100.0,8060.923477
14,oppo a6 5g,2,6343.197088,19150.0,12806.802912



--- Worst 20 Unseen Listings ---


,title,brand,model_key_norm,price,predicted_price,absolute_error
240,Huawei Mate X5 ميتx5,huawei,huawei mate x5,50000.0,23952.514501,26047.485499
128,Sony Xperia PRO-I موبايل سوني إكسبيريا برو,sony,sony xperia pro-i,45543.0,24664.841918,20878.158082
213,إنفنيكس نوت 60,infinix,infinix note 60,22000.0,7804.556798,14195.443202
354,شاومي 8 برو ماكس متفتحش زى الفل الشاشه وكله با...,xiaomi,xiaomi 8 pro,6000.0,19038.806585,13038.806585
307,Google Pixel 9 Pro,google,google pixel 9 pro,32000.0,21066.083741,10933.916259
358,سامسونج a27,samsung,samsung a27,18000.0,7689.885016,10310.114984
112,إنفنيكس نوت 5,infinix,infinix note 5,12500.0,2252.805326,10247.194674
133,إنفنيكس نوت 60,infinix,infinix note 60,16500.0,7992.010617,8507.989383
286,اوبو ار 15 برو,oppo,oppo r15 pro,1000.0,8722.234325,7722.234325
476,شاومي بوكو اف 6,xiaomi,xiaomi poco f6,17000.0,24503.108787,7503.108787


In [7]:
# Unseen-model error analysis by title_group
unseen_group_report = unseen_df.groupby('title_group', dropna=False).agg(
    ads_count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    median_actual_price=('price', 'median'),
    median_predicted_price=('predicted_price', 'median')
).reset_index().sort_values('MAE', ascending=False)

print("Number of unseen title groups:", len(unseen_group_report))

display(unseen_group_report)

Number of unseen title groups: 40


,title_group,ads_count,MAE,median_actual_price,median_predicted_price
3,huawei mate x5 ميتx5,1,26047.485499,50000.0,23952.514501
8,sony xperia pro i موبايل سوني اكسبيريا برو,1,20878.158082,45543.0,24664.841918
21,شاومي 8 برو ماكس متفتحش زي الفل الشاشه وكله با...,1,13038.806585,6000.0,19038.806585
14,انفنيكس نوت 60,2,11351.716292,19250.0,7898.283708
1,google pixel 9 pro,1,10933.916259,32000.0,21066.083741
19,سامسونج a27,1,10310.114984,18000.0,7689.885016
13,انفنيكس نوت 5,1,10247.194674,12500.0,2252.805326
16,اوبو ار 15 برو,1,7722.234325,1000.0,8722.234325
22,شاومي بوكو اف 6,2,6961.834213,16500.0,23461.834213
30,موبايل سامسونج جلاكسي جراند,1,6960.923477,1100.0,8060.923477


In [8]:
output_path = "/content/task7_3_reports.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    brand_report.to_excel(writer, sheet_name="MAE by Brand", index=False)
    price_report.to_excel(writer, sheet_name="MAE by Price Band", index=False)
    frequency_report.to_excel(writer, sheet_name="MAE by Frequency", index=False)
    unseen_model_report.to_excel(writer, sheet_name="Unseen Models", index=False)
    unseen_group_report.to_excel(writer, sheet_name="Unseen Title Groups", index=False)
    worst_unseen.to_excel(writer, sheet_name="Worst 20 Unseen", index=False)

    summary = pd.DataFrame({
        "Metric": [
            "Total evaluated phone listings",
            "Overall Final Model MAE",
            "Unseen-model listings",
            "Distinct unseen models",
            "Unseen-model MAE",
            "Distinct unseen title groups"
        ],
        "Value": [
            len(report_df),
            report_df["absolute_error"].mean(),
            len(unseen_df),
            unseen_df["model_key_norm"].nunique(),
            unseen_df["absolute_error"].mean(),
            unseen_df["title_group"].nunique()
        ]
    })

    summary.to_excel(writer, sheet_name="Summary", index=False)

print("Task 7.3 reports saved successfully!")
files.download(output_path)

Task 7.3 reports saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>